In [5]:
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
df = pd.read_csv("/IMDB Dataset.csv.zip")

df.shape
df.head()
df.info()
df.describe()
df['sentiment'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


,count
sentiment,
positive,25000
negative,25000


In [23]:
df.rename(columns={'review': 'text'}, inplace=True)
df = df.sample(10000, random_state=42)

In [25]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]

    return " ".join(words)

df['clean_text'] = df['text'].apply(preprocess)

In [27]:
df[['text', 'clean_text']].head()

,text,clean_text
25056,"the tortuous emotional impact is degrading, wh...",tortuou emot impact degrad whether adult adole...
30334,Anyone who knows anything about evolution woul...,anyon know anyth evolut even need see film say...
17962,I'm glad I rented this movie for one reason: i...,glad rent movi one reason shortcom made want r...
39588,"Yes, the votes are in. This film may very well...",ye vote film may well plan outer space gener w...
34107,This mini-series is actually more entertaining...,mini seri actual entertain other much bigger b...


In [29]:
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

In [32]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])
y = df['sentiment'].map({'positive':1, 'negative':0})
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

In [34]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [36]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [38]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

In [40]:
def evaluate(y_test, y_pred, model_name):
    print(f"Model: {model_name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("\n")

In [42]:
evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")

Model: Logistic Regression
Accuracy: 0.8725
              precision    recall  f1-score   support

           0       0.90      0.84      0.87      1008
           1       0.85      0.90      0.88       992

    accuracy                           0.87      2000
   macro avg       0.87      0.87      0.87      2000
weighted avg       0.87      0.87      0.87      2000



Model: Naive Bayes
Accuracy: 0.855
              precision    recall  f1-score   support

           0       0.86      0.86      0.86      1008
           1       0.85      0.85      0.85       992

    accuracy                           0.85      2000
   macro avg       0.85      0.85      0.85      2000
weighted avg       0.85      0.85      0.85      2000



Model: Decision Tree
Accuracy: 0.702
              precision    recall  f1-score   support

           0       0.71      0.68      0.70      1008
           1       0.69      0.72      0.71       992

    accuracy                           0.70      2000
   macro

## 📊 Model Comparison & Insights

- **Logistic Regression** delivered the best results due to its ability to handle high-dimensional TF-IDF features efficiently.  

- **Naive Bayes** was computationally efficient but slightly less accurate.  

- **Decision Tree** tended to overfit, making it less reliable for generalization.  

- **TF-IDF** outperformed Bag of Words as it assigns importance to meaningful words rather than just frequency.